# Translated-hybrid vs dense candidates — fresh frozen evidence v1

48 cases,12 paired intents. Questions/labels/corpus unchanged. Existing E5 document AND query vectors reused.
Automatic translations were made once with the existing Groq gpt-oss-20b QueryTranslator (auto language detection),
with explicit user consent. All24 succeeded. No API key or database access is needed on Colab.

Production translated_hybrid.search is reused through a frozen-corpus adapter: original-query dense,
language-matched BM25 from original/automatic-translated queries, RRF k60, weights dense1/lexical2.
Verified document languages come from the frozen snapshot. Internal branch budget300 (production top_k*10),
fused reranker candidates30, final10. PostgreSQL simple lexemes/production cleanup/formula are replayed offline;
BM25 score parity passed, but stable chunk-ID tie breaks replace unavailable live DB UUIDs.

Both dense/plain and translated-hybrid/plain are reranked AGAIN on this same Colab GPU, FP32/pinned BGE.
Shared candidate-query inputs are scored once with exact lineage. Historical control agreement is diagnostic.
No predecessor context, translation of passages, answer generation, production change or label tuning.
Report BOTH candidate30 availability and final10 evidence, plus regressions. This is development data, not held-out proof.


In [ ]:
%pip -q install sentence-transformers==6.0.1 transformers==5.17.0 huggingface-hub==1.31.0 numpy==2.4.6 python-dotenv openai==2.41.0 pydantic==2.11.7


In [ ]:
import os
os.environ.update(OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',TOKENIZERS_PARALLELISM='false',HF_HUB_DISABLE_XET='1')
import torch
assert torch.cuda.is_available(), 'Select GPU runtime.'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()  # Select translated_fresh_colab_inputs.zip.
assert 'translated_fresh_colab_inputs.zip' in uploaded
work=Path('/content/translated_fresh_experiment')
assert not work.exists(), 'Already extracted: continue below or choose a new directory.'
work.mkdir()
with zipfile.ZipFile('translated_fresh_colab_inputs.zip') as z:
    assert all((work/n).resolve().is_relative_to(work.resolve()) for n in z.namelist())
    z.extractall(work)
del uploaded
os.chdir(work)


In [ ]:
import sys,subprocess
runner=[sys.executable,'scripts/retrieval/evaluate_translated_fresh_reranker.py']
subprocess.run(runner+['--preflight'],check=True)


In [ ]:
output=Path('/content/translated_fresh_results')
# Reuses cached BGE weights if available. No Groq key, DB or E5 query inference needed.
subprocess.run(runner+['--output-dir',str(output),'--allow-download'],check=True)


In [ ]:
subprocess.run(runner+['--output-dir',str(output),'--verify-only'],check=True)
import json
report=json.loads((output/'results.json').read_text())
print('Direction | Dense+rerank | Translated-hybrid+rerank | Dense pool30 | Hybrid pool30')
for d in ('vi->vi','en->en','vi->en','en->vi'):
    counts=[]
    for a in ('dense','translated_hybrid'):
        s=report['summary'][a][d];counts.append(f"{s['full_evidence_cases_at_10']}/{s['full_intent_eligible_cases']}")
    for a in ('dense_candidate30','translated_hybrid_candidate30'):
        s=report['summary'][a][d];counts.append(f"{s['full_evidence_cases_at_30']}/{s['full_intent_eligible_cases']}")
    print(d,*counts,sep=' | ')


In [ ]:
import shutil
archive=shutil.make_archive('/content/translated_fresh_results','zip',root_dir=output)
files.download(archive)
# Return this ZIP to Codex. Partial/failed outputs are not completed benchmark results.
